# `microbetag`: metabolic secrets behind microbial co-occurrence

## How to use this notebook

This notebook runs on a **GitHub Codespace**, so you don’t have to worry about setup.

`microbetag` though has two main versions to use: 

- one we call *on-the-fly* since it is running on a virtual machine at KU Leuven and the user (us!) only uses Cytoscape and the MGG add-on to communicater with it. 
- locally, i.e. 

When using `microbetag` locally, things can get trickier because it comes with a lot of dependencies. But don’t worry — you only need the ones for the features you actually use.

For example, `microbetag` supports two ways of working with Genome-Scale Metabolic Reconstructions (GEMs). If you’re only interested in one approach, you can skip the dependencies for the other.




> GitHub Codespace: a cloud-based online integrated development environment developed by GitHub. It allows users to create and manage development environments directly within the browser or through Visual Studio Code desktop.

In [1]:
!git clone https://github.com/msysbio/microbetag

Cloning into 'microbetag'...
remote: Enumerating objects: 4115, done.
remote: Counting objects: 100% (805/805), done.
remote: Compressing objects: 100% (381/381), done.
remote: Total 4115 (delta 488), reused 591 (delta 418), pack-reused 3310 (from 2)
Receiving objects: 100% (4115/4115), 292.89 MiB | 31.93 MiB/s, done.
Resolving deltas: 100% (2032/2032), done.
Updating files: 100% (503/503), done.


In [ ]:
https://bactopia.github.io/latest/

In [1]:
import os

### Get a Gurobi license

In case you wish to reconstruct a GEM, either with ModelSEEDpy or CarveMe, or gap-fill one using DNNGIOR, in all these cases you need to have first get yourself a Gurobi license. 

Gurobi is **commercial** software (sorry for this) 
yet, there is a free license for academic purposes. 

If you wish to run this notebook locally, you need to make sure you get one first! You may [follow our instructions](https://github.com/hariszaf/metabolic_toy_model/blob/duth/prep_env.ipynb) for how to do this from a previous course. 




In [2]:

# Create directory for the license
os.makedirs("licenses", exist_ok=True)

# Function to make sure you can use Gurobi on Colab
def create_gurobi_license(WLSACCESSID, WLSSECRET, LICENSEID):
    license_content = (
        "# Gurobi WLS license file\n"
        "# Your credentials are private and should not be shared or copied to public repositories.\n"
        "# Visit https://license.gurobi.com/manager/doc/overview for more information.\n"
        f"WLSACCESSID={WLSACCESSID}\n"
        f"WLSSECRET={WLSSECRET}\n"
        f"LICENSEID={LICENSEID}"
    )
    with open("licenses/gurobi.lic", "w") as f:
        f.write(license_content)
    print("License file created at licenses/gurobi.lic")



In [3]:
# WLSACCESSID="d5419c87-0d36-4a93-9385-773f5483b3c1"
# WLSSECRET="afa5d95f-ad0b-4a38-9550-a8913aacb7c0"
# LICENSEID="964844"

WLSACCESSID="cc3447b4-ad86-4542-98e3-3213c37b799e"
WLSSECRET="51a40c73-6f0d-49eb-866b-9966111a2e86"
LICENSEID="964844"

In [4]:
create_gurobi_license(WLSACCESSID, WLSSECRET, LICENSEID)

License file created at licenses/gurobi.lic


In [5]:


import gurobipy as gbp

model = gbp.Model("test")
print("Gurobi is working!", "\U0001F600")



ModuleNotFoundError: No module named 'gurobipy'

## Our data 

For this tutorial, we will use two different test cases

| Use case | Description | Purpose |
|----------|-------------|---------|
| **Synthetic community (4-species)** | A certain species is believed to have a positive effect on several others | Use `microbetag` locally with **our own genomes** to explore the concept of **complementarity** in detail |
| **Natural communities (hundreds of taxa)** | Co-occurrence network derived from communities across multiple samples | Use `microbetag` annotations with **network clustering** and **enrichment analysis** to **generate new hypotheses** on community drivers |


### Synthetic community

The following list of **obligate anaerobes**:

- *Anaerococcus vaginalis* (Av)
- *Finegoldia magna* (Fm)
- *Peptoniphilus asaccharolyticus* (Pa)

have been found to have a positive association with *Pseudomonas aeruginosa* (Pseud), a **facultative anaerobe**.

For all those four species, the online version of `microbetag` already has at least a GTDB representative genome, however, we will download their corresponding genomes so we can run an example of how one would use `microbetag` with their own, custom genomes.

To this end, let's have a look at the celebrated [GTDB](https://gtdb.ecogenomic.org/).


You may see, that there are several genomes from most of those genera, and in many cases of the exact species -- since we have no further information on what strains have this association we have to *guess* 🤷🏾  

| Strain | GTDB entry | NCBI entry | 
|:------:|:----------:|:----------:|
| Pseudomonas aeruginosa |  [GCF_001457615.1](https://gtdb.ecogenomic.org/genome?gid=GCF_001457615.1) | [GCA_001457615.1](https://www.ncbi.nlm.nih.gov/datasets/genome/GCA_001457615.1) |
| Finegoldia magna_H |  [GCF_000010185.1](https://gtdb.ecogenomic.org/genome?gid=GCF_000010185.1) | [GCA_000010185.1](https://www.ncbi.nlm.nih.gov/datasets/genome/GCA_000010185.1) |
| Anaerococcus vaginalis |  [GCF_000311745.1](https://gtdb.ecogenomic.org/genome?gid=GCF_000311745.1) | [GCA_000311745.1](https://www.ncbi.nlm.nih.gov/datasets/genome/GCA_000311745.1) |
| Peptoniphilus asaccharolyticus |  [GCF_900176115.1](https://gtdb.ecogenomic.org/genome?gid=GCF_900176115.1) | [GCA_900176115.1](https://www.ncbi.nlm.nih.gov/datasets/genome/GCA_900176115.1) |


One can get genomes, using their [NCBI Rest API](https://www.ncbi.nlm.nih.gov/datasets/docs/v2/api/rest-api/), 

<!-- curl -X GET "https://api.ncbi.nlm.nih.gov/datasets/v2/genome/accession/GCF_001457615.1/download?include_annotation_type=GENOME_FASTA" -H 'accept: application/zip' 

curl -L -o genome.zip "https://api.ncbi.nlm.nih.gov/datasets/v2/genome/accession/GCF_001457615.1/download?include_annotation_type=GENOME_FASTA" -H 'accept: application/zip' -->

```
curl -L -o genomes.zip "https://api.ncbi.nlm.nih.gov/datasets/v2/genome/accession/GCF_001457615.1%2CGCF_000010185.1%2CGCF_000311745.1%2CGCF_900176115.1/download?include_annotation_type=GENOME_FASTA" -H 'accept: application/zip'
```



from the **Bacterial and Viral Bioinformatics Resource Center (BV-BRC)** [platform](https://www.bv-brc.org/).


In [ ]:
genomes = {
    "Pseud": "GCF_001457615.1",
    "Fm"   : "GCF_000010185.1",
    "Av"   : "GCF_000311745.1",
    "Pa"   : "GCF_900176115.1"
}

## Metabolic modeling 

For a thorough basic intro, you may have a look on another [branch](https://github.com/hariszaf/metabolic_toy_model/tree/duth) of this repo, 
where you may check how to deal with a model as a Python object using the [`cobra` library](https://cobrapy.readthedocs.io/), as weel as several constraint-based methods.